# Day 25 — DENSE Repository Understanding + First Run

## Goal

Today we move from **paper understanding** to **code understanding**.

The target is not to understand every file. The target is to answer:

$$
\text{Paper module}
\rightarrow
\text{Repo file/function}
\rightarrow
\text{Runnable command}
$$

By the end of Day 25, you should know where the main entry point, dataset preparation, bundle sampling, LLM query, losses, GNN training, refinement, and evaluation live in the code.

Official repository: https://github.com/YushengZhao/bundle-neurips25

> Today is allowed to include a **first run attempt**. A failed run with a clear error log still counts as progress.

## 1. Repo map — first 10 minutes

Open the repository root and fill in:

- `dataset/`:
> Stores the datasets used by DENSE.

- `model/`:
> Contains implementations of GNN models.

- `resources/`:
> Stores resources used by the README, such as framework figures.

- `bundle.py`:
> Main entry point for running DENSE experiments.

- `queryhelper.py`:
> Likely handles LLM querying. Need to verify from source code.

- `utils.py`:
> Likely contains utility/helper functions. Need to verify from source code.

- `README.md`:
> Provides environment setup, data setup, API setup, and experiment commands.

### Q1
Which file is the main entry point?

> bundle.py

### Q2
What evidence from the README supports your answer?

> The README runs all experiments through `python bundle.py` with different datasets and configurations, indicating that `bundle.py` is the main entry point.

## 2. Map paper modules to repo locations

Fill this table while browsing the code.

| Paper component | Repo file / function | What it does |
|---|---|---|
| Bundle sampling |bundle.py → bundle_sample() |Samples a core node and constructs a bundle using topological or feature proximity |
| Bundle query |bundle.py → bundle_query() + queryhelper.py → QueryHelper.query() |Collects node texts in a bundle, queries the LLM, and converts its response into a bundle label |
| Bundle supervision |bundle.py → bundle_loss() |Uses bundle labels as supervision to train the GNN |
| GNN model |utils.py → prepare_model() |Constructs the trainable GNN model, currently GCN or GloGNN/MLP_NORM. |
| $L_{BE}$ |bundle.py → bundle_loss() (loss_type='average') |Averages node logits within each bundle and applies cross-entropy against the bundle label |
| $L_R$ |bundle.py → bundle_loss() (loss_type='ranking') |Penalizes the model when the bundle-label class is ranked below the highest-probability class |
| Bundle refinement |Not verifiable in this file |solve() calls bundle_resample(), but no implementation of bundle_resample() appears in the provided file |
| Evaluation |bundle.py → evaluate() |Predicts node classes with the trained GNN and computes classification accuracy on test/validation nodes |

Do not guess. Record only what you can verify from the code.

## 3. Dataset preparation

The repository's `dataset/` directory may not contain the real datasets directly.

Read the README and record:

### Dataset source / preparation instructions

> Cora

### Which dataset will we try first?

Recommended first choice:

> **Cora**, if the official instructions make it easy to prepare.

### Why choose this dataset first?

> Cora is a relatively small and standard citation-network dataset, so it is a good first choice for verifying whether the repository can run successfully.

### Q3
What files must exist before `bundle.py` can run?

> For Cora, the repository expects at least:

- `dataset/cora/processed_data.pt`
- `dataset/cora/raw_texts.pt`
- `dataset/cora/categories.csv`
- `dataset/cora/llmicl_class_aware_x.pt`

## 4. Environment reconnaissance

Before installing anything, inspect the code imports and README.

Record the likely dependencies:

- Python version:
> Python 3.8.18

- PyTorch:
> PyTorch 2.1.2

- PyTorch Geometric:
> torch_geometric==2.5.0

- OpenAI client:
> openai (version not specified)

- NumPy / SciPy / scikit-learn:
> NumPy is used in the code. SciPy and scikit-learn are not explicitly specified in the README.

- Other dependencies:
> torchvision==0.16.2, torchaudio==2.1.2, DGL==2.4.0+cu121, pyg_lib==0.3.1+pt21cu121, torch_scatter==2.1.2, torch_sparse==0.6.18+pt21cu121, torch_cluster==1.6.3+pt21cu121, torch_spline_conv==1.2.2, transformers==4.46.3, sentence_transformers==2.2.2, protobuf, accelerate, pandas.

### Q4
Does the repo provide a complete `requirements.txt` or environment file?

> No. The dependencies are provided as installation commands in the README instead.

### Q5
What dependency is most likely to cause installation trouble on your machine?

> The CUDA-dependent PyTorch Geometric / DGL stack is most likely to cause installation trouble, because the README specifies Linux and CUDA 12.1-specific package versions, while my current development environment is Windows.

## 5. Main command anatomy

From the README, copy one official command for a small dataset such as Cora.

```bash
python bundle.py --device 0 --dataset cora --bundle_size 5 --num_samples 100 --sample_criterion neighbor --max_hop 2 --query_type gpt --model gpt-4o --loss_type ranking --gnn_type gin --stages 400 100 100 --valid --lr 0.001 --wd 0.001 --resample --repeat 1
```

Explain the important arguments:

- `--dataset`:
> Selects the dataset to run on. Here it is Cora.
- `--bundle_size`:
> Number of nodes included in each bundle. Here each bundle contains 5 nodes.
- `--num_samples`:
> Number of bundles sampled. Here DENSE samples 100 bundles.
- `--sample_criterion`:
> Determines how nodes are selected to construct bundles. neighbor constructs bundles according to graph/topological proximity.
- `--max_hop`:
> Maximum graph-hop distance considered when sampling neighboring nodes. Here it is 2 hops.
- `--query_type`:
> Determines the type of query mechanism. Here gpt means using a GPT-based LLM query.
- `--model`:
> Specifies the LLM used for bundle queries. Here it is GPT-4o.
- `--loss_type`:
> Specifies the supervision loss. Here ranking selects the ranking-based supervision setting.
- `--gnn_type`:
> Specifies the GNN backbone used for node prediction. Here the official Cora command uses GIN.
- `--stages`:
> Specifies the number of training epochs/iterations for different training or refinement stages. For Cora, the stages are 400 100 100.
- `--lr`:
> Learning rate used to optimize the GNN. Here it is 0.001.
- `--wd`:
> Weight decay used for regularization during optimization. Here it is 0.001.

### Q6
Which of these are **research variables** and which are mostly **training / engineering settings**?

> Research variables: bundle_size, num_samples, sample_criterion, max_hop, query_type, model, loss_type, and gnn_type.
> Training / engineering settings: device, stages, lr, wd, valid, resample, and repeat.

## 6. API dependency

The README requires an OpenAI API key.

### Important

Do **not** put your real API key into:
- notebook cells
- README
- Git commits
- screenshots

Prefer an environment variable, for example in PowerShell:

```powershell
$env:OPENAI_API_KEY="YOUR_KEY"
```

### Q7
Which LLM does the official command use?

> GPT-4o

### Q8
Does the experiment query the LLM online during every run, or can it reuse saved query results?

> It can reuse cached query results. The code first checks whether the current prompt exists in the cache. If it does, the cached response is returned; otherwise, the OpenAI API is called and the new response is stored in the cache.

## 7. Find the loss implementation

Search the code for:

```text
ranking
entropy
cross_entropy
loss_type
L_BE
L_R
```

### $L_{BE}$ implementation location

> `bundle.py -> bundle_loss()`, in the `loss_type == 'average'` branch.
> The code averages the node logits inside each bundle and applies cross-entropy loss against the bundle label.

### $L_R$ implementation location

> `bundle.py -> bundle_loss()`, in the `loss_type == 'ranking'` branch.
> The code computes the probability of the LLM-provided bundle class, compares it with the maximum class probability, and applies a ranking penalty when the bundle class is not top-ranked.

### Q9
Can you identify where the bundle-level logits are aggregated before computing the loss?

> Yes. In the `average` branch, node-level logits are aggregated by
> `average_logits = torch.mean(bundle_logits, dim=1)`.
> In the `ranking` branch, the implementation first applies softmax to node logits and then averages the node-level probabilities with
> `bundle_prob_mean = torch.mean(bundle_prob, dim=1)`.

### Q10
Does the implementation match the paper formula conceptually?

> Partially. The `average` branch conceptually matches the bundle-level cross-entropy loss $L_{BE}$: it averages node logits within a bundle and applies cross-entropy using the bundle label. The ranking branch also captures the main idea of $L_R$: it penalizes the model when the LLM-provided bundle class is not the highest-ranked class. However, the exact aggregation in the ranking branch differs from the paper formula, because the code averages per-node softmax probabilities, while the paper defines the bundle probability by applying softmax after averaging the logits. The shown ranking branch also contains a `cross_entropy` call without an explicit target argument, which should be verified before running the code.

## 8. Find bundle refinement

Search for keywords such as:

```text
refine
refinement
evict
remove
preserve
stage
```

### File / function

> `bundle.py -> bundle_resample()` appears to correspond to the bundle refinement step described in the paper. However, in the current repository snapshot, `bundle_resample()` is called in `solve()` but no function definition is present.

### Trigger timing

The paper says refinement happens during optimization. How is it scheduled in code?

> It is scheduled between training stages. After each `bundle_optimize(...)` stage, except the final stage, the code calls `bundle_resample(...)` before continuing to the next stage.

### Q11
What criterion appears to determine whether a node is removed / preserved?

> Cannot be determined from the current repository snapshot, because the implementation of `bundle_resample()` is missing. The code does not expose the refinement criterion here, so it should not be inferred from the paper alone.

> Repo note: the current code snapshot appears incomplete or inconsistent with the paper/README, because the refinement routine is referenced but not implemented.

## 9. First run attempt

Today we are allowed to try a first run.

Before running, check:

- [ ] dataset prepared
- [ ] environment created
- [ ] dependencies installed
- [ ] API key available if needed
- [x] command copied from README
- [x] no real API key is stored in the repo

### Run command

```bash
# The official training command has not been executed yet.
# Preflight check:
nvidia-smi
```

### Result

- [ ] Success
- [x] Failed during environment setup
- [ ] Failed during dataset loading
- [ ] Failed during LLM query
- [ ] Failed during GNN training
- [ ] Other

### Error / output

```text
zsh: command not found: nvidia-smi
```

### Q12
What is the **first blocking issue**?

> The first blocking issue is the execution environment. My local machine does not provide a usable NVIDIA GPU/CUDA environment, while the official repository setup is designed around Linux, PyTorch 2.1.2, CUDA 12.1, PyG, and DGL. Therefore, I should prepare a compatible Linux GPU environment before attempting the full experiment.

## 10. Minimal reproduction plan for Day 26

Based on today's repo reading, define the smallest experiment we should run tomorrow.

### Dataset
> Cora

### GNN
> GCN

### LLM / query mode
> GPT-based bundle query using GPT-4o, preferably using cahed response if available.

### Bundle size
> 5

### Number of bundles
> a small number first, e.g. 20–50, for debugging; then increase toward the official setting after the pipeline works

### Epochs / stages
> Use a reduced staged schedule for the first debugging run, for example `10 5 5`, instead of immediately using the full `400 100 100`

### Evaluation metric
> Node classification accuracy on the test split.

### Why is this the smallest meaningful baseline?
> It preserves the essential DENSE pipeline: graph-based bundle construction, LLM-generated bundle supervision, GNN training, staged optimization, and final node-level evaluation. At the same time, it reduces the number of bundles and training epochs so that environment, data-loading, API, and code issues can be diagnosed quickly before running the full configuration.

## 11. Paper ↔ Code mapping

### Paper says: "bundle sampling"

Code does:
> `bundle.py -> bundle_sample()`

### Paper says: "bundle query"

Code does:
> `bundle.py -> bundle_query()` and `batch_bundle_query()`, with LLM interaction handled by `QueryHelper`

### Paper says: "bundle supervision"

Code does:
> `bundle.py -> bundle_loss()`

### Paper says: "bundle refinement"

Code does:
> `solve()` calls `bundle_resample()` between optimization stages, which appears to correspond to bundle refinement. However, the implementation of `bundle_resample()` is missing from the current repository snapshot.

### Paper says: "zero-shot node prediction"

Code evaluates:
> `bundle.py -> evaluate()`, which performs node classification using the trained GNN and reports accuracy on the evaluation split.

## 12. Researcher's reflection

### Q13
Which part of the repo is easiest to understand because of Day 23–24 paper reading?

> The bundle sampling → bundle query → bundle supervision pipeline is the easiest part to understand. Because I already understood the paper-level roles of these components, functions such as `bundle_sample()`, `bundle_query()`, and `bundle_loss()` can be directly mapped to the method described in the paper.

### Q14
Which part of the code is still confusing?

> The staged optimization and bundle refinement logic is still the most confusing part. In particular, `bundle_resample()` is called between training stages but its implementation is missing from the current repository snapshot, so I cannot yet determine exactly how nodes are removed, preserved, or resampled.

### Q15
What implementation detail did the paper make look simpler than it really is?

> Bundle refinement looked like a simple conceptual step in the paper, but the code reveals that it must interact with staged GNN training, bundle states, re-querying, and resampling. The implementation also contains several practical details, such as caching LLM responses, handling invalid bundle queries, scheduling multiple training stages, and matching tensor shapes for bundle-level losses.

### Q16
If you had to modify only one variable later for a controlled experiment, which one would you choose now?

> I would choose `bundle_size`. It directly affects how much structural/contextual information is included in each bundle, can be changed without modifying the main algorithm, and provides a clean single-variable experiment for studying the trade-off between richer bundle context and increased label noise.

## Day 25 final summary

**Your answer:**

> DENSE is mainly organized around `bundle.py`, which controls the main experiment pipeline. The major paper components, including bundle sampling, bundle query, bundle supervision, GNN optimization, and evaluation, can be mapped to functions such as `bundle_sample()`, `bundle_query()`, `bundle_loss()`, `bundle_optimize()`, and `evaluate()`. The LLM interaction is handled by `queryhelper.py`, while `utils.py` is responsible for loading data, graph features, and preparing the GNN model. The repository requires external graph datasets, an OpenAI API key for GPT-based bundle queries, and a Linux/PyTorch/CUDA/PyG/DGL environment. The code also supports caching previous LLM query results to reduce repeated API calls. However, the current repository has several inconsistencies, especially that `bundle_resample()` is called between training stages but its implementation is missing. My local machine also has no usable NVIDIA GPU, so the required CUDA environment cannot currently be prepared locally. Therefore, the next step is to prepare a Linux GPU environment, download the Cora dataset, install the required dependencies, and first run a small debugging baseline before attempting the full official experiment.

# Day 25 Completion Checklist

- [x] Identified the main entry file
- [x] Mapped paper modules to repo files/functions
- [x] Understood dataset preparation requirements
- [x] Identified important dependencies
- [x] Understood the official command
- [x] Located LLM/API usage
- [x] Located the two loss implementations
- [ ] Located refinement logic
- [x] Attempted at least one run OR documented the exact blocker
- [x] Defined a minimal Day 26 baseline reproduction plan

**Stop when you know exactly what must happen next to get one baseline result.**